# PyMC-NMR Tutorial 1: Getting Started with Basin Hopping

Welcome to **PyMC-NMR**! This tutorial will walk you through the process of setting up and running a structure search using the modern **YAML configuration** and **Python API**.

## Objectives
1. Prepare an initial structure file (`basin.xyz`) using ASE.
2. Set up the unified simulation configuration (`config_tutorial.yaml`).
3. Execute the simulation using the pythonic `run_simulation` API.
4. Inspect output files and log results.

### Step 1: Prepare the Structure File (`basin.xyz`)

First, we read our starting structure and save it as `basin.xyz`.

In [ ]:
from ase.io import read, write

# Read starting structure and apply substitution
atoms = read("CHA_SAR5_with_H.extxyz")
write("basin.xyz", atoms, format="extxyz")
print("Created basin.xyz successfully!")

> **Why do the hydrogens look wrong?**
>
> In `CHA_SAR5_with_H.extxyz` each proton was placed on a sphere 2.5 Å from an
> Al, without being attached to any oxygen, so the nearest O is ~2.3-2.5 Å away
> rather than the ~0.98 Å of a real Brønsted O-H bond. This is an intentionally
> crude starting guess. The **first geometry relaxation seats them correctly**:
> all six protons move onto Al-O-Si bridges at O-H = 0.97-1.00 Å, lowering the
> energy by ~23 eV. Nothing is broken.


### Hardware: use a GPU if you possibly can

Every basin-hopping step evaluates the MACE potential many times (once per
relaxation step), so **runtime is dominated by the potential, and a GPU is
roughly one to two orders of magnitude faster than a CPU.** On a CPU a single
basin-hopping step of this 114-atom cell takes minutes, so a realistic run of a
few hundred steps is an overnight job at best.

Set the device in the `potential` block:

```yaml
potential:
  device: cuda     # NVIDIA GPU  (use "mps" on Apple Silicon, "cpu" otherwise)
```

Check that PyTorch can actually see your GPU first:

```python
import torch
print(torch.cuda.is_available())   # must be True for device: cuda
```

If that prints `False`, you have a CPU-only PyTorch build and must reinstall a
CUDA-enabled one, otherwise the run will fail. The config below uses `cpu` with
only a handful of steps so that this tutorial finishes in a reasonable time;
switch to `cuda` and raise `monte.steps` for production.


### Step 2: Write the Configuration File (`config_tutorial.yaml`)

We now write our configuration directly in Python. 
Notice that **you do not need to list elements or chemical species in the config file!** PyMC-NMR automatically derives them from your structure file.

We write to `config_tutorial.yaml` rather than `config.yaml` so that this
short tutorial run does not overwrite `config_production.yaml`, the realistic
long-running configuration shipped alongside it.


In [ ]:
import yaml

config_data = {
    "job": {
        "structure_path": "basin.xyz",
        "structure_method": "basinhop",
        "temperature": 1200.0,
        "dump_archive": True,
        "archive_frequency": 20,
        "save_downhill": True,
        "equil_steps": 50
    },
    "potential": {
        "arch": "mace_mp",
        "model": "MACE-matpes-r2scan-omat-ft.model",
        "device": "cpu",
        "precision": "float64"
    },
    "monte": {
        "steps": 10
    },
    "moves": {
        "swaps": [
            {
                "pairs": [["Al", "Si"]],
                "counterion": "H",
                "max_distance": 4.0,
                "frequency": 100
            }
        ]
    },
    "relax": {
        "method": "bfgs",
        "steps": 500,
        "tol": 0.01
    }
}

with open("config_tutorial.yaml", "w") as f:
    yaml.safe_dump(config_data, f, default_flow_style=False)
print("Written config_tutorial.yaml successfully!")

### Step 3: Run the Simulation via Python API

We can import and run the simulation directly from Python without running clunky terminal subprocesses.

In [ ]:
from pymc_nmr.monte import run_simulation

# Run the structure search directly within Jupyter!
run_simulation(config_path="config_tutorial.yaml", structure_path="basin.xyz", log_path="mc.log")


### Step 4: Understanding Output Files

Once the simulation runs, you will find these files in your directory:
* **mc.log**: Text file detailing calculation parameters, accepted swaps, and energies.
* **restart.xyz**: The final, optimized structure.
* **archive.xyz**: Structural trajectory history.